In [ ]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import soundfile as sf
import torchaudio.compliance.kaldi as kaldi
import torch.nn.functional as torch_f
# ==============================================================================
# 1. GPU ACCELERATED EXTRACTOR (FULL TORCH)
# ==================================================    ============================
class FastFeatureExtractor:
    def __init__(self, device, sample_rate=16000, n_mfcc=40, n_mels=80, n_fft=400, hop_length=160):
        self.device = device
        self.sr = sample_rate
        self.hop = hop_length
        self.n_fft = n_fft

        # MFCC và MelSpectrogram trả về: [Batch, Channel, Freq, Time]
        self.mfcc_transform = T.MFCC(
            sample_rate=sample_rate, n_mfcc=n_mfcc,
            melkwargs={"n_fft": n_fft, "n_mels": n_mels, "hop_length": hop_length, "center": False}
        ).to(device)

        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=n_fft, n_mels=n_mels, 
            hop_length=hop_length, center=False
        ).to(device)

    def _apply_cmvn(self, x):
        # Tính toán mean/std trên chiều Time (dim=-1)
        return (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-6)

    @torch.no_grad()
    def _compute_pitch_gpu(self, waveforms):
        """ Tính Pitch tự động trên GPU với shape [B, 1, 1, T] để khớp MFCC """
        # [B, 1, T] -> [B, Frames, N_FFT]
        frames = waveforms.squeeze(1).unfold(-1, self.n_fft, self.hop)
        
        # FFT Autocorrelation
        fft = torch.fft.rfft(frames, n=self.n_fft)
        autocorr = torch.fft.irfft(fft.abs().pow(2), n=self.n_fft)
        
        # Khoảng tìm kiếm F0 (60Hz - 500Hz)
        min_dist, max_dist = int(self.sr / 500), int(self.sr / 60)
        search_area = autocorr[:, :, min_dist:max_dist]
        max_indices = torch.argmax(search_area, dim=-1) + min_dist
        
        pitch = self.sr / (max_indices.float() + 1e-6)
        # Thêm 2 chiều để thành [B, 1, 1, Frames] khớp với [B, 1, Freq, Frames] của MFCC
        return pitch.unsqueeze(1).unsqueeze(1)

    @torch.no_grad()
    def extract_batch(self, waveforms):
        # Cả 2 đều trả về [B, 1, Freq, Time]
        mfcc = self.mfcc_transform(waveforms)
        mel = self.mel_transform(waveforms)
        mfbe = torch.log(mel + 1e-6)

        # Pitch: [B, 1, 1, Time]
        pitch = self._compute_pitch_gpu(waveforms)

        # Align chiều Time
        min_len = min(mfcc.shape[-1], pitch.shape[-1], mfbe.shape[-1])
        mfcc, mfbe, pitch = mfcc[..., :min_len], mfbe[..., :min_len], pitch[..., :min_len]

        # Cat theo chiều Frequency (dim=2)
        results = {
            "Only_MFBE": self._apply_cmvn(mfbe),
            "Only_Pitch": self._apply_cmvn(pitch),
            "MFBE_Pitch": self._apply_cmvn(torch.cat([mfbe, pitch], dim=2))
        }
        # Squeeze các chiều dư thừa trước khi trả về CPU để tiết kiệm RAM
        return {k: v.squeeze(1).cpu() for k, v in results.items()}
# ==============================================================================
# 2. TỐI ƯU DATA LOADER
# ==============================================================================
class AudioDataset(Dataset):
    def __init__(self, root_dir, sample_rate=16000):
        self.root_dir = root_dir
        self.sr = sample_rate
        self.file_list = [os.path.join(r, f) for r, _, files in os.walk(root_dir) 
                          for f in files if f.lower().endswith(('.wav', '.flac'))]
        self.resampler = {} # Cache resamplers for different source SRs

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]
        try:
            wav, sr = torchaudio.load(path)
            if wav.shape[0] > 1: wav = torch.mean(wav, dim=0, keepdim=True)
            if sr != self.sr:
                wav = T.Resample(sr, self.sr)(wav)
            return wav, path
        except: return None

def fast_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    waves, paths = zip(*batch)
    
    # Pad waveforms để cùng độ dài trong 1 batch trước khi đẩy lên GPU
    max_len = max([w.shape[-1] for w in waves])
    padded_waves = torch.stack([torch.nn.functional.pad(w, (0, max_len - w.shape[-1])) for w in waves])
    return padded_waves, paths

# ==============================================================================
# 3. MAIN EXECUTION
# ==============================================================================
INPUT_PATH = r"D:\Study\7-SP26\DATxSLP\Test set O\test_o_clean"
OUTPUT_BASE_PATH = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after extract feature"
device = torch.device("cuda")

extractor = FastFeatureExtractor(device)
dataset = AudioDataset(INPUT_PATH)
loader = DataLoader(
    dataset, 
    batch_size=8, # Tăng batch_size lên (4060 8GB có thể chịu được 64-128)
    shuffle=False, 
    num_workers=0, # Sử dụng đa nhân CPU để load file từ disk
    pin_memory=True, # Tăng tốc độ copy dữ liệu lên GPU
    collate_fn=fast_collate
)

all_features = {m: {} for m in ["Only_MFCC", "Only_MFBE", "Only_Pitch", "MFCC_Pitch", "MFBE_Pitch"]}

print(f"🚀 Processing with {torch.cuda.get_device_name(0)} (Single-threaded loading)...")

for batch in tqdm(loader):
    if batch is None: continue
    waveforms, paths = batch
    
    # Đẩy nguyên batch lớn lên GPU
    # non_blocking=True giúp chồng lấn việc copy dữ liệu và tính toán
    waveforms = waveforms.to(device, non_blocking=True)
    
    # Trích xuất toàn bộ batch (Hàm dùng FFT GPU đã sửa lỗi dimension ở trên)
    batch_results = extractor.extract_batch(waveforms)
    
    # Lưu kết quả
    for i, path in enumerate(paths):
        rel_path = os.path.relpath(path, INPUT_PATH)
        for mode in batch_results.keys():
            all_features[mode][rel_path] = batch_results[mode][i]

# Save
os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)
for mode, data in all_features.items():
    torch.save(data, os.path.join(OUTPUT_BASE_PATH, f"{mode}.pt"))
    print(f"✅ Saved {mode}")

🚀 Processing with NVIDIA GeForce RTX 4060 Laptop GPU (Single-threaded loading)...


100%|██████████| 2224/2224 [01:09<00:00, 32.03it/s]


✅ Saved Only_MFCC
✅ Saved Only_MFBE
✅ Saved Only_Pitch
✅ Saved MFCC_Pitch
✅ Saved MFBE_Pitch
